# APPLES

### IMPORT

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from arch import arch_model
from spread import SPREAD
from screener import SCREENER
from engine import ENGINE
from backtester import BACKTESTER
from tearsheet import TEARSHEET

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath('../scripts'))

### DATA

In [2]:
months = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/eurnok_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/eurnok_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/eursek_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/eursek_dukascopy_bid_{m}.parquet" for m in months],
]

### SPREAD

In [ ]:
builder = SPREAD(agg_type='volume', threshold=1000, active_hours=(10, 14))
df = builder.build(my_files)

print(df.head(3))
print("\nColumns:", list(df.columns))
print(f"Median half-spread (bps) — A: {df['HalfSpread_A_bps'].median():.2f} | "
      f"B: {df['HalfSpread_B_bps'].median():.2f}")

### SCREENER

In [ ]:
screener = SCREENER(df['Asset_A'], df['Asset_B'])
p_val, hl = screener.generate_report()

### ENGINE

In [ ]:
live_trading_data, df_params = ENGINE.walk_forward(df, train_days=30, z_window=250)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(live_trading_data.index, live_trading_data['Beta'],
        color='tab:blue', linewidth=0.9, label='Rolling Beta (OOS)')
ax.axhline(live_trading_data['Beta'].mean(), color='black', linestyle='--',
           alpha=0.6, label=f"Mean = {live_trading_data['Beta'].mean():.3f}")
ax.set_title("Rolling OLS Hedge Ratio (OOS)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Beta range: [{live_trading_data['Beta'].min():.3f}, {live_trading_data['Beta'].max():.3f}]")
print(f"Beta std:    {live_trading_data['Beta'].std():.4f}")

### BACKTESTER

In [ ]:
import importlib
import backtester  

importlib.reload(backtester)
from backtester import BACKTESTER

bt = BACKTESTER(live_trading_data)
results_df = bt.run(
    base_z=1.25,
    exit_z=0.0,
    danger_threshold=0.7,          # fixed: was 1.1 which never triggered
    ar_limit=0.995,
    fee_bps=0.5,
    stop_loss_bps=100.0,
    take_profit_bps=25.0,
    max_hold_bars=150,
    slippage_mode='half_spread',   # realistic execution cost
)

### TEARSHEET

In [ ]:
ts = TEARSHEET(results_df, df_params=df_params)
ts.generate_report()
ts.plot_comparative_equity()
ts.plot_engine_parameters()

In [ ]:
heat_df = ts.plot_robustness_heatmap(
    engine_data=live_trading_data,
    base_z_grid=np.round(np.arange(0.75, 2.51, 0.25), 2),
    danger_grid=np.round(np.arange(0.3, 0.96, 0.1), 2),
    fixed_params={
        'exit_z': 0.0,
        'ar_limit': 0.995,
        'fee_bps': 0.5,
        'stop_loss_bps': 100.0,
        'take_profit_bps': 25.0,
        'max_hold_bars': 150,
        'slippage_mode': 'half_spread',
    },
    metric='sharpe',
)
print("\nTop 5 (base_z, danger_threshold) by Sharpe:")
print(heat_df.stack().sort_values(ascending=False).head(5))

In [ ]:
# Quick A/B: does rolling cointegration materially change the adaptive PnL
# vs the old static full-sample fit? Run one pass with the static method and
# compare Sharpe. This cell is optional; it demonstrates the old approach.

def run_static_pipeline():
    rows = []
    for i in range(train_days, len(unique_days)):
        train_dates = unique_days[i - train_days : i]
        test_date = unique_days[i]
        tr = df[df['Date'].isin(train_dates)].copy()
        te = df[df['Date'] == test_date].copy()
        if len(tr) <= Z_WINDOW + 50:
            continue
        eng = ENGINE(tr)
        eng.fit_cointegration_static(z_window=Z_WINDOW)  # <-- the old behavior
        eng.fit_ar_reversion(lags=1)
        eng.fit_garch_vol(scaling=10000)
        eng.fit_markov_regimes(k_regimes=2)
        oos = eng.predict_oos(
            test_df=te, train_tail_df=eng.data, z_window=Z_WINDOW,
            coint_window=COINT_WINDOW, coint_refit_every=1,
            refit_every=500, garch_window=5000, ar_window=2000,
        )
        rows.append(oos)
    return pd.concat(rows)

# NOTE: predict_oos still uses rolling coint across the boundary; to make this
# a true apples-to-apples against a pure static approach, set coint_window
# very large (effectively expanding) or write a short static-OOS helper.
# For now, treat this as an illustration of the API, not a clean benchmark.